# Stage 1 global576 LoRA — official benchmark evaluation

Matched baseline: **576 Projection-B global tokens only** + LoRA.
No Projection-A, RAM++, or Grounding DINO.

- VQAv2 test-dev: EvalAI submission (private labels)
- POPE: macro/per-split F1
- MMBench EN dev: diagnostic accuracy + VLMEvalKit export
- SEED-Bench image: dimensions 1–9 (not video-inclusive “SEED All”)

GQA is intentionally excluded (same rationale as the Stage 3 eval notebook).

Reusable helpers live in `evaluation.py`; JSONL predictions resume automatically.

## Setup Environment

In [1]:
import os
import sys
from pathlib import Path

# Set these before importing any `reva` modules. Update the placeholder paths
# to match your machine or shared Jupyter environment.
os.environ["REVA_DATA_ROOT"] = "/path/to/your/data/root"
os.environ["REVA_HF_CACHE_ROOT"] = "/path/to/your/hf_cache/root"
os.environ["REVA_CHECKPOINT_ROOT"] = "/path/to/your/checkpoints/root"
os.environ["REVA_EVAL_RESULTS_ROOT"] = "/path/to/your/eval_results/root"
os.environ["REVA_REGION_DATA_ROOT"] = "/path/to/your/region_data/root"
os.environ["REVA_DECONTAMINATION_ROOT"] = "/path/to/your/decontamination/root"
os.environ["REVA_GROUNDING_DINO_ROOT"] = "/path/to/your/groundingdino/root"
os.environ["REVA_VQAV2_ROOT"] = "/path/to/your/vqav2/root"
os.environ["REVA_TEST_IMAGES_ROOT"] = "/path/to/your/test_images/root"

hf_cache_root = Path(os.environ["REVA_HF_CACHE_ROOT"]).expanduser()
os.environ["HF_HOME"] = str(hf_cache_root)
os.environ["HF_HUB_CACHE"] = str(hf_cache_root / "hub")
os.environ["HF_DATASETS_CACHE"] = str(hf_cache_root / "datasets")
os.environ["TRANSFORMERS_CACHE"] = str(hf_cache_root / "hub")

for var_name in (
    "REVA_DATA_ROOT",
    "REVA_HF_CACHE_ROOT",
    "REVA_CHECKPOINT_ROOT",
    "REVA_EVAL_RESULTS_ROOT",
    "REVA_REGION_DATA_ROOT",
    "REVA_DECONTAMINATION_ROOT",
    "REVA_GROUNDING_DINO_ROOT",
    "REVA_VQAV2_ROOT",
    "REVA_TEST_IMAGES_ROOT",
    "HF_HOME",
    "HF_HUB_CACHE",
    "HF_DATASETS_CACHE",
    "TRANSFORMERS_CACHE",
):
    print(f"{var_name} = {os.environ.get(var_name)}")


def find_reva_project_root(start: Path) -> Path:
    override = os.environ.get("REVA_PROJECT_DIR")
    if override:
        return Path(override).expanduser().resolve()

    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "reva" / "evaluation.py").is_file() and (candidate / "reva" / "config.py").is_file():
            return candidate

    raise FileNotFoundError(
        "Could not locate the ReVA project root from the current working directory. "
        "Set REVA_PROJECT_DIR to your cloned repo path."
    )


PROJECT_ROOT = find_reva_project_root(Path.cwd())

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.chdir(PROJECT_ROOT)
print("Working directory:", Path.cwd())


REVA_DATA_ROOT = /path/to/your/data/root
REVA_HF_CACHE_ROOT = /path/to/your/hf_cache/root
REVA_CHECKPOINT_ROOT = /path/to/your/checkpoints/root
REVA_EVAL_RESULTS_ROOT = /path/to/your/eval_results/root
REVA_REGION_DATA_ROOT = /path/to/your/region_data/root
REVA_DECONTAMINATION_ROOT = /path/to/your/decontamination/root
REVA_GROUNDING_DINO_ROOT = /path/to/your/groundingdino/root
REVA_VQAV2_ROOT = /path/to/your/vqav2/root
REVA_TEST_IMAGES_ROOT = /path/to/your/test_images/root
HF_HOME = /path/to/your/hf_cache/root
HF_HUB_CACHE = /path/to/your/hf_cache/root/hub
HF_DATASETS_CACHE = /path/to/your/hf_cache/root/datasets
TRANSFORMERS_CACHE = /path/to/your/hf_cache/root/hub
Working directory: /home/jovyan/reva-main


In [ ]:
import os
from pathlib import Path

# Must run before importing transformers / huggingface_hub.
from reva.config import configure_hf_cache

HF_CACHE = configure_hf_cache(os.environ["REVA_HF_CACHE_ROOT"])
print("HF cache root:", HF_CACHE)
print("HF hub cache:", HF_CACHE / "hub")
print("Free in /tmp:", f"{os.statvfs('/tmp').f_bavail * os.statvfs('/tmp').f_frsize / 1e9:.0f}G")

os.environ.setdefault("USE_TF", "0")
os.environ.setdefault("USE_FLAX", "0")
os.environ.setdefault("USE_TORCH", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    free_gb, total_gb = torch.cuda.mem_get_info()
    print(f"GPU mem free: {free_gb / 1e9:.1f} / {total_gb / 1e9:.1f} GB")

## Paths and run controls

In [ ]:
from pathlib import Path

PROJECT_DIR = Path(".").resolve()
DATA_ROOT = Path(os.environ.get("REVA_DATA_ROOT") or os.path.expanduser("~/reva-data"))
PROJECTION_B = PROJECT_DIR / "projection_b_best_weights.pt"

# Point this at your completed Stage 1 run's best_lora directory.
LORA = Path("/home/jovyan/teaching_material/stage1_lora")
OUTPUT_DIR = DATA_ROOT / "eval_results/global576_stage1"

LIMIT = None              
RESUME = True  # False deletes selected benchmark's old JSONL

RUN_VQAV2 = True
RUN_POPE = False
RUN_MMBENCH = False
RUN_SEED_IMAGE = False

for path in (PROJECTION_B, LORA):
    assert path.exists(), path
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Outputs:", OUTPUT_DIR)

## Load global576 stack

In [ ]:
from reva.evaluation import (
    evaluate_mmbench_full,
    evaluate_pope_full,
    evaluate_seed_image_full,
    evaluate_vqav2_testdev_full,
    load_stage1_eval_stack,
)

stack = load_stage1_eval_stack(
    projection_b_path=PROJECTION_B,
    lora_path=LORA,
)
print("Loaded on", stack.config.device, "|", stack.config.compute_dtype)
print("Visual prefix: 576 global tokens only (0 region tokens)")

## Run selected benchmarks

In [ ]:
%%bash
set -euo pipefail

# One-time setup
cd /tmp
git clone https://github.com/open-compass/VLMEvalKit.git
cd VLMEvalKit
pip install -e .
pip install rouge-score

EVAL_ROOT="${REVA_EVAL_RESULTS_ROOT}"
MMB_DIR="$EVAL_ROOT/global576_stage1"

# Rename so VLMEvalKit infers dataset name (must end with MMBench_DEV_EN)
cp "$MMB_DIR/MMBench_DEV_EN_with_predictions.tsv" \
   "$MMB_DIR/global576_stage1_MMBench_DEV_EN.tsv"

# Official MMBench circular eval (exact matching, no GPT judge)
cd /tmp/VLMEvalKit
vlmutil eval "$MMB_DIR/global576_stage1_MMBench_DEV_EN.tsv" \
  --judge exact_matching

# Read score (Overall row = reportable number)
cat "$MMB_DIR/global576_stage1_MMBench_DEV_EN_acc.csv"

In [ ]:
summaries = {}

if RUN_POPE:
    summaries["pope"] = evaluate_pope_full(
        stack, pope_root=DATA_ROOT / "pope",
        coco_val2014_dir=DATA_ROOT / "coco/val2014", output_dir=OUTPUT_DIR,
        limit=LIMIT, resume=RESUME,
    )

if RUN_MMBENCH:
    summaries["mmbench"] = evaluate_mmbench_full(
        stack, mmbench_root=DATA_ROOT / "mmbench", output_dir=OUTPUT_DIR,
        split="MMBench_DEV_EN", limit=LIMIT, resume=RESUME,
    )

if RUN_SEED_IMAGE:
    summaries["seed_image"] = evaluate_seed_image_full(
        stack, seed_root=DATA_ROOT / "seed_bench", output_dir=OUTPUT_DIR,
        limit=LIMIT, resume=RESUME,
    )

if RUN_VQAV2:
    summaries["vqav2_testdev"] = evaluate_vqav2_testdev_full(
        stack, vqav2_root=DATA_ROOT / "vqav2", output_dir=OUTPUT_DIR,
        limit=LIMIT, resume=RESUME,
    )

## Report and save combined summary

In [ ]:
import json

for name, result in summaries.items():
    if "accuracy_pct" in result:
        print(f"{name:16s}: {result['accuracy_pct']:.2f}% (n={result['n']})")
    elif "macro_f1_pct" in result:
        print(f"{name:16s}: F1={result['macro_f1_pct']:.2f}% (n={result['n']})")
    elif "local_accuracy_pct" in result:
        print(f"{name:16s}: local={result['local_accuracy_pct']:.2f}% (canonical via VLMEvalKit)")
    else:
        print(f"{name:16s}: {result['n']} predictions; external scoring required")

combined_path = OUTPUT_DIR / "all_summaries.json"
payload = {
    "experiment": "stage1_global576_lora",
    "lora_path": str(LORA),
    "projection_b_path": str(PROJECTION_B),
    "visual_input": "576 global tokens only",
    "summaries": summaries,
}
with open(combined_path, "w") as f:
    json.dump(payload, f, indent=2)
print("Saved:", combined_path)
print("VQAv2 submission:", OUTPUT_DIR / "vqav2_testdev_submission.json")